In [5]:
# Cell: Imports and setup
import pandas as pd
import numpy as np
import pydicom
import cv2
import os
import shutil

data_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train = pd.read_csv(f"{data_dir}/train.csv")
train_series = pd.read_csv(f"{data_dir}/train_series.csv")

finding_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 
                    'Contusion', 'Fracture']

gold_data = train[train['Fracture'].notna()]
gold_labels = gold_data[['StudyInstanceUID'] + finding_columns].copy()
v2_labels = pd.read_csv('/kaggle/input/datasets/mayankptdr/v2-labels/extracted_labels_v2.csv')
all_labels_v2 = pd.concat([gold_labels, v2_labels], ignore_index=True)

print(f"Total studies: {all_labels_v2.shape[0]}")

Total studies: 4407


In [6]:
# Helper functions — series-selection, mm-crop, resize, RGB-grouping, physical-ordering
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1)]

def select_series_5slots(study_id, series_df):
    study_series = series_df[series_df['StudyInstanceUID'] == study_id]
    slots = {}
    for plane, fluid_sensitive in SLOTS:
        key = f"{plane}_{fluid_sensitive}"
        candidates = study_series[
            (study_series['Anatomical_Plane'] == plane) & 
            (study_series['Fluid_Sensitive'] == fluid_sensitive)
        ]
        slots[key] = candidates.iloc[0]['SeriesInstanceUID'] if len(candidates) > 0 else None
    return slots

def crop_by_mm(pixel_array, pixel_spacing, crop_mm=130.0):
    row_spacing, col_spacing = pixel_spacing
    crop_h = int(crop_mm / row_spacing)
    crop_w = int(crop_mm / col_spacing)
    h, w = pixel_array.shape
    center_h, center_w = h // 2, w // 2
    half_h, half_w = crop_h // 2, crop_w // 2
    top = max(0, center_h - half_h)
    bottom = min(h, center_h + half_h)
    left = max(0, center_w - half_w)
    right = min(w, center_w + half_w)
    return pixel_array[top:bottom, left:right]

def crop_and_resize(pixel_array, pixel_spacing, crop_mm=130.0, final_size=224):
    cropped = crop_by_mm(pixel_array, pixel_spacing, crop_mm)
    resized = cv2.resize(cropped, (final_size, final_size), interpolation=cv2.INTER_LINEAR)
    return resized

def select_slice_indices_grouped(total_slices, n_groups=16, band=(0.12, 0.88)):
    start = int(total_slices * band[0])
    end = int(total_slices * band[1])
    center_indices = np.linspace(start, end - 1, n_groups).astype(int)
    groups = []
    for center in center_indices:
        prev_idx = max(0, center - 1)
        next_idx = min(total_slices - 1, center + 1)
        groups.append((prev_idx, center, next_idx))
    return groups

def load_rgb_group(slices_sorted, group_indices, pixel_spacing, crop_mm=130.0, final_size=224):
    channels = []
    for idx in group_indices:
        arr = slices_sorted[idx].pixel_array.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
        cr = crop_and_resize(arr, pixel_spacing, crop_mm, final_size)
        channels.append(cr)
    return np.stack(channels, axis=-1)

def get_position_key(d):
    try:
        orientation = [float(x) for x in d.ImageOrientationPatient]
        row_vec = np.array(orientation[:3])
        col_vec = np.array(orientation[3:])
        normal = np.cross(row_vec, col_vec)
        position = np.array([float(x) for x in d.ImagePositionPatient])
        return np.dot(position, normal)
    except Exception:
        return int(d.InstanceNumber)

print("Helper functions ready")

Helper functions ready


In [7]:
# Chunk-splitting
CHUNK_NUMBER = 1

all_study_ids = all_labels_v2['StudyInstanceUID'].tolist()
chunk_size = len(all_study_ids) // 10

chunks = {}
for i in range(1, 11):
    start_i = (i-1) * chunk_size
    end_i = i * chunk_size if i < 10 else len(all_study_ids)
    chunks[i] = all_study_ids[start_i:end_i]

my_chunk = chunks[CHUNK_NUMBER]
print(f"Chunk {CHUNK_NUMBER}: {len(my_chunk)} studies assigned")

Chunk 1: 440 studies assigned


In [8]:
# Cache-building — RGB-grouped, float16
cache_dir_v2 = "/kaggle/working/cached_v2"
os.makedirs(cache_dir_v2, exist_ok=True)

def build_cache_v2(study_ids, n_groups=16, crop_mm=130.0, final_size=224):
    for study_id in study_ids:
        cache_path = f"{cache_dir_v2}/{study_id}.npz"
        if os.path.exists(cache_path):
            continue
        
        total, used, free = shutil.disk_usage("/kaggle/working")
        if free / (1024**3) < 1.0:
            print(f"Disk space low. Stopping safely.")
            break
        
        try:
            slot_map = select_series_5slots(study_id, train_series)
            slot_volumes, slot_masks = [], []
            
            for plane, fs in SLOTS:
                key = f"{plane}_{fs}"
                series_id = slot_map[key]
                
                if series_id is None:
                    vol = np.zeros((n_groups, final_size, final_size, 3), dtype=np.float16)
                    mask = 0.0
                else:
                    folder = f"{data_dir}/train_series/{study_id}/{series_id}"
                    dcm_files = os.listdir(folder)
                    slices = [pydicom.dcmread(f"{folder}/{f}") for f in dcm_files]
                    slices_sorted = sorted(slices, key=get_position_key)
                    total_slices = len(slices_sorted)
                    pixel_spacing = [float(x) for x in slices_sorted[0].PixelSpacing]
                    groups = select_slice_indices_grouped(total_slices, n_groups)
                    
                    rgb_groups = []
                    for group_indices in groups:
                        rgb = load_rgb_group(slices_sorted, group_indices, pixel_spacing, crop_mm, final_size)
                        rgb_groups.append(rgb)
                    vol = np.stack(rgb_groups).astype(np.float16)
                    mask = 1.0
                
                slot_volumes.append(vol)
                slot_masks.append(mask)
            
            slots_array = np.stack(slot_volumes)
            mask_array = np.array(slot_masks, dtype=np.float32)
            np.savez_compressed(cache_path, slots=slots_array, mask=mask_array)
            print(f"Cached: {study_id}")
        
        except Exception as e:
            print(f"Error on {study_id}: {e}")

build_cache_v2(my_chunk)

Cached: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Cached: 1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269
Cached: 1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052
Cached: 1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580
Cached: 1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238
Cached: 1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935
Cached: 1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840
Cached: 1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756
Cached: 1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604
Cached: 1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581
Cached: 1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945
Cached: 1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819
Cached: 1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973
Cached: 1.2.826.0.1.3680043.8.498.12801308844398614

In [9]:
print("completed")

completed
